# Neuron-Cluster Circuit Distillation

End-to-end pipeline for distilling arithmetic reasoning from a Llama-3-8B
teacher into a Llama-3.2-1B student by aligning **neuron-cluster-level**
representations discovered via the circuit discovery framework.

**Pipeline overview:**
1. Load pre-computed neuron clusters from k-means
2. Run ablation study per cluster on both student and teacher
3. Pair clusters between models by functional importance
4. Train the student with a composite loss: `L = L_CE + λ · L_cluster_CKA`

**Prerequisites:** neuron clustering results in `results/neuron-clustering/`,
a trained circuit-discovery checkpoint, and the 2-digit addition datasets.

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate huggingface_hub tqdm matplotlib

## 2. Imports and Setup

In [ ]:
import os
import sys
import json
import re
import random
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import login

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("datasets", exist_ok=True)
os.makedirs("results/cluster-distillation", exist_ok=True)

## 3. HuggingFace Login

In [ ]:
HF_TOKEN = ""

if HF_TOKEN:
    login(HF_TOKEN)
else:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            login(token=hf_token)
            print("Logged in using Colab secret HF_TOKEN")
        else:
            login()
    except Exception:
        login()

## 4. Configuration

In [ ]:
TEACHER_MODEL = "meta-llama/Meta-Llama-3-8B"
STUDENT_MODEL = "meta-llama/Llama-3.2-1B"

CIRCUIT_CHECKPOINT = "epoch_4000.pt"  # circuit-discovery checkpoint
K_CLASSES = 8                         # latent subclasses in circuit model

# k value (number of neuron clusters) used for each subclass.
# Adjust if you ran neuron clustering with different k per subclass.
CLASS_CLUSTERS_STUDENT = [6] * K_CLASSES
CLASS_CLUSTERS_TEACHER = [6] * K_CLASSES

# Training hyperparameters
EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
LAMBDA_CLUSTER = 0.01          # weight for cluster-CKA alignment loss
LAMBDA_PROJ = 0.0              # weight for optional projection loss (0 = off)
GRAD_CLIP = 1.0
TOP_K_CLUSTERS = 5             # top-k most important cluster pairs per subclass
IMPORTANCE_WEIGHTING = True    # weight CKA loss by ablation importance
USE_PROJECTION_HEADS = False   # set True to add learned projection heads
EVAL_EVERY = 1

# Paths
TRAIN_DATASET_PATH = "datasets/2d_add_train_80.json"
TEST_DATASET_PATH = "datasets/2d_add_test_20.json"
SAVE_DIR = "results/cluster-distillation"

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f"Student: {STUDENT_MODEL}")
print(f"Teacher: {TEACHER_MODEL}")
print(f"Subclasses: {K_CLASSES}, clusters/subclass: {CLASS_CLUSTERS_STUDENT[0]}")
print(f"Top-k cluster pairs: {TOP_K_CLUSTERS}")
print(f"lambda_cluster: {LAMBDA_CLUSTER}, LR: {LEARNING_RATE}")

## 5. Load Datasets

In [ ]:
def find_file(filename):
    """Search common paths for the given filename."""
    candidates = [
        filename,
        f"datasets/{filename}",
        os.path.basename(filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Cannot find {filename}")

TRAIN_DATA = json.load(open(find_file(TRAIN_DATASET_PATH)))
TEST_DATA = json.load(open(find_file(TEST_DATASET_PATH)))

print(f"Train: {len(TRAIN_DATA)} examples")
print(f"Test:  {len(TEST_DATA)} examples")
print(f"Sample: {list(TRAIN_DATA.items())[0]}")

## 6. Circuit Discovery Model Classes

Copied from the neuron-clustering notebook so this notebook is self-contained.
The circuit discovery model classifies problems into latent subclasses and
produces neuron masks — we use it here to (a) load neuron masks and
(b) pre-classify training data.

In [ ]:
config_1b = AutoConfig.from_pretrained(STUDENT_MODEL)
config_8b = AutoConfig.from_pretrained(TEACHER_MODEL)
model_configs = {"1b": config_1b, "8b": config_8b}


class ProblemEncoder(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.op1_emb_layer = nn.Embedding(100, embedding_dim // 4)
        self.op2_emb_layer = nn.Embedding(100, embedding_dim // 4)
        self.sum_emb_layer = nn.Embedding(200, embedding_dim // 2)

    def forward(self, op1, op2, res):
        return torch.cat((
            self.op1_emb_layer(op1),
            self.op2_emb_layer(op2),
            self.sum_emb_layer(res),
        ), dim=-1)


class ProblemClassifier(nn.Module):
    def __init__(self, input_dim, k_classes, hidden1=256, hidden2=32):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
            nn.Linear(hidden2, k_classes),
        )

    def forward(self, x):
        return self.classifier(x)


class NeuronMask(nn.Module):
    def __init__(self, k_classes, activations_dim):
        super().__init__()
        hidden_dim = 4
        self.k_classes = k_classes
        self.class_embedding = nn.Embedding(k_classes, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, activations_dim)

    def forward(self, class_probs, activations):
        class_ids = class_probs.argmax(dim=-1)
        hidden = F.relu(self.class_embedding(class_ids))
        sigmoid_mask = torch.sigmoid(self.output_layer(hidden))
        masked = activations * sigmoid_mask.unsqueeze(1)
        return masked, sigmoid_mask

    def class_masks(self):
        class_ids = torch.arange(self.k_classes, device=self.class_embedding.weight.device)
        hidden = F.relu(self.class_embedding(class_ids))
        return torch.sigmoid(self.output_layer(hidden))


class CircuitDiscoveryModel(nn.Module):
    def __init__(self, k_classes, problem_embedding_dim=256, tau=0.5):
        super().__init__()
        self.tau = tau
        num_act_1b = model_configs["1b"].intermediate_size * model_configs["1b"].num_hidden_layers
        num_act_8b = model_configs["8b"].intermediate_size * model_configs["8b"].num_hidden_layers

        self.problem_encoder = ProblemEncoder(embedding_dim=problem_embedding_dim)
        self.classifier = ProblemClassifier(problem_embedding_dim, k_classes)
        self.neuron_masks_1b = NeuronMask(k_classes, num_act_1b)
        self.neuron_masks_8b = NeuronMask(k_classes, num_act_8b)

    def classify_problem(self, op1, op2, res):
        return self.classifier(self.problem_encoder(op1, op2, res))

    def forward(self, op1, op2, res, activations_1b, activations_8b):
        logits = self.classify_problem(op1, op2, res)
        hard = F.gumbel_softmax(logits, tau=self.tau, hard=True, dim=-1)
        masked_1b, mask_1b = self.neuron_masks_1b(hard, activations_1b)
        masked_8b, mask_8b = self.neuron_masks_8b(hard, activations_8b)
        return {
            "hard_class_probs": hard,
            "masked_activations_1b": masked_1b,
            "masked_activations_8b": masked_8b,
            "mask_1b": mask_1b, "mask_8b": mask_8b,
        }

print("Circuit discovery model classes defined.")

## 7. Utility Functions

In [ ]:
def _safe_model_name(name: str) -> str:
    return name.replace("/", "_").replace(":", "_")


def parse_equation(probs, device=None):
    op1_list, op2_list, res_list = [], [], []
    for prob in probs:
        add_idx = prob.index("+")
        eq_idx = prob.index("=")
        op1_list.append(int(prob[:add_idx]))
        op2_list.append(int(prob[add_idx + 1 : eq_idx]))
        res_list.append(int(prob[eq_idx + 1 :]))
    return (
        torch.tensor(op1_list, dtype=torch.long, device=device),
        torch.tensor(op2_list, dtype=torch.long, device=device),
        torch.tensor(res_list, dtype=torch.long, device=device),
    )


def load_circuit_checkpoint(path, k_classes, lr=1e-3):
    checkpoint = torch.load(path, map_location=device)
    model = CircuitDiscoveryModel(k_classes=k_classes).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    epoch = checkpoint.get("epoch", checkpoint.get("step", 0))
    print(f"Loaded circuit-discovery checkpoint from epoch {epoch}")
    return model

## 8. Load Neuron Clusters

Load the pre-computed k-means neuron cluster assignments produced by
`neuron_clustering.ipynb`.  For each model and subclass, the cluster file
contains a mapping from cluster id to a tensor of flattened neuron indices.

In [ ]:
def load_all_clusters(model_name, class_clusters, base_dir="results/neuron-clustering"):
    """Load cluster assignments for all subclasses of a model.

    Returns:
        Dict[subclass_int] -> Dict[cluster_int] -> Tensor of neuron indices
    """
    safe = _safe_model_name(model_name)
    clusters_dir = os.path.join(base_dir, safe)
    all_clusters = {}

    for sc in range(len(class_clusters)):
        k = class_clusters[sc]
        path = os.path.join(clusters_dir, f"subclass_{sc}_clusters/k{k}.pt")
        if not os.path.exists(path):
            print(f"  [WARN] Missing cluster file: {path}")
            continue
        ckpt = torch.load(path, map_location="cpu")
        c2i = ckpt["cluster_to_indices"]
        all_clusters[sc] = c2i
        sizes = [c2i[c].numel() for c in c2i]
        print(f"  Subclass {sc}: {len(c2i)} clusters, sizes={sizes}")

    return all_clusters


print("Student clusters:")
student_clusters = load_all_clusters(STUDENT_MODEL, CLASS_CLUSTERS_STUDENT)

print("\nTeacher clusters:")
teacher_clusters = load_all_clusters(TEACHER_MODEL, CLASS_CLUSTERS_TEACHER)

## 9. Neuron-Cluster Ablation Study

For each model, ablate one cluster at a time and record the accuracy drop
on that subclass's problems. This establishes per-cluster importance
scores used to pair clusters between student and teacher.

**Note:** This is the most expensive step (loads the full LLM many times).
Results are cached to `ablation_performance.json` and reused on re-runs.

In [ ]:
# ---- Ablation helper functions (self-contained in notebook) ----

def apply_ablation(model, neuron_indices):
    """Zero out specific neurons (by flattened MLP index) in a Llama-style model."""
    cfg = model.config
    intermediate_size = cfg.intermediate_size
    num_layers = cfg.num_hidden_layers
    layers = model.model.layers

    if isinstance(neuron_indices, torch.Tensor):
        idx_list = neuron_indices.view(-1).tolist()
    else:
        idx_list = list(neuron_indices)

    with torch.no_grad():
        for idx in idx_list:
            idx = int(idx)
            layer_id = idx // intermediate_size
            neuron_id = idx % intermediate_size
            if layer_id < 0 or layer_id >= num_layers:
                continue
            mlp = layers[layer_id].mlp
            for proj in [mlp.gate_proj, mlp.up_proj]:
                if 0 <= neuron_id < proj.weight.shape[0]:
                    proj.weight[neuron_id].zero_()
            if 0 <= neuron_id < mlp.down_proj.weight.shape[1]:
                mlp.down_proj.weight[:, neuron_id].zero_()
    return model


def _eval_on_dataset(model, tokenizer, dataset_records, batch_size=50):
    """Evaluate a model on a list of {q_str, a_str} records.  Returns accuracy."""
    model.eval()
    prompts = [r["q_str"] for r in dataset_records]
    answers = [r["a_str"] for r in dataset_records]

    correct = total = 0
    for i in range(0, len(prompts), batch_size):
        bp = prompts[i : i + batch_size]
        ba = answers[i : i + batch_size]
        inputs = tokenizer(bp, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        for pred_text, gold in zip(decoded, ba):
            m = re.search(r"=\s*(\d+)", pred_text)
            if m and m.group(1) == gold:
                correct += 1
            total += 1
    return correct / max(total, 1)


def run_cluster_ablation(
    model_name, tokenizer, class_to_problems, all_clusters, class_clusters,
    results_dir=None,
):
    """Run full ablation study: per-cluster accuracy drop for every subclass.

    Returns and caches ablation_performance.json under results_dir.
    """
    safe = _safe_model_name(model_name)
    if results_dir is None:
        results_dir = os.path.join("results", "circuit-discovery", safe)
    os.makedirs(results_dir, exist_ok=True)

    out_path = os.path.join(results_dir, "ablation_performance.json")
    if os.path.exists(out_path):
        print(f"Loading cached ablation results from {out_path}")
        with open(out_path) as f:
            return json.load(f)

    ablation_results = {}

    for sc in range(len(class_clusters)):
        sc_str = str(sc)
        problems = class_to_problems.get(sc_str, [])
        if not problems or sc not in all_clusters:
            continue

        # Build subclass-specific eval dataset
        records = []
        for prob_str, _ in problems:
            if "=" not in prob_str:
                continue
            lhs, rhs = prob_str.split("=", 1)
            ans = rhs.strip()
            if not ans.isdigit():
                continue
            records.append({"q_str": lhs + "=", "a_str": ans})

        if not records:
            continue

        # Baseline
        print(f"\nSubclass {sc}: evaluating baseline ({len(records)} problems)...")
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else None,
        ).to(device)
        base_model.eval()
        baseline_acc = _eval_on_dataset(base_model, tokenizer, records)
        print(f"  Baseline accuracy: {baseline_acc:.4f}")
        del base_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        cluster_accs = {}
        for cid, neuron_idx in all_clusters[sc].items():
            abl_model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else None,
            ).to(device)
            abl_model.eval()
            apply_ablation(abl_model, neuron_idx)
            acc = _eval_on_dataset(abl_model, tokenizer, records)
            cluster_accs[str(cid)] = acc
            print(f"  Cluster {cid}: accuracy {acc:.4f}  (drop: {baseline_acc - acc:+.4f})")
            del abl_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        ablation_results[sc_str] = {"baseline": baseline_acc, "clusters": cluster_accs}

    with open(out_path, "w") as f:
        json.dump(ablation_results, f, indent=2)
    print(f"\nSaved ablation results to {out_path}")
    return ablation_results

### 9a. Classify all problems into subclasses

The circuit model assigns every 2-digit addition problem to a latent subclass.
We need these assignments to create per-subclass evaluation sets for ablation.

In [ ]:
CLASSIFIED_PATH = "results/circuit-discovery/classified_problems.json"

if os.path.exists(CLASSIFIED_PATH):
    with open(CLASSIFIED_PATH) as f:
        class_to_problems = json.load(f)
    print(f"Loaded cached classifications from {CLASSIFIED_PATH}")
else:
    circuit_model = load_circuit_checkpoint(CIRCUIT_CHECKPOINT, K_CLASSES)
    circuit_model.eval()

    # Load full 2-digit addition dataset
    with open("datasets/2d_add_all.json") as f:
        all_problems = json.load(f)

    ids = torch.tensor([r["ids"] for r in all_problems]).to(device)
    prompts = tokenizer.batch_decode(ids, skip_special_tokens=True)

    class_to_problems = {}
    for i in range(0, len(prompts), 256):
        bp = prompts[i : i + 256]
        bi = ids[i : i + 256]
        op1, op2, res = parse_equation(bp, device=device)
        with torch.no_grad():
            logits = circuit_model.classify_problem(op1, op2, res)
            subclass = torch.argmax(logits, dim=-1)
        for prob, cls, tid in zip(bp, subclass.tolist(), bi.tolist()):
            class_to_problems.setdefault(str(cls), []).append((prob, tid))

    os.makedirs(os.path.dirname(CLASSIFIED_PATH), exist_ok=True)
    with open(CLASSIFIED_PATH, "w") as f:
        json.dump(class_to_problems, f, indent=2)
    print(f"Saved classifications to {CLASSIFIED_PATH}")

    del circuit_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for k in sorted(class_to_problems, key=int):
    print(f"  Subclass {k}: {len(class_to_problems[k])} problems")

### 9b. Run ablation for student and teacher

This cell runs the ablation study. On first run it will load each model
many times (once per cluster per subclass). Results are cached to JSON.

In [ ]:
print("=" * 50)
print("STUDENT ABLATION")
print("=" * 50)
student_ablation = run_cluster_ablation(
    STUDENT_MODEL, tokenizer, class_to_problems,
    student_clusters, CLASS_CLUSTERS_STUDENT,
)

print("\n" + "=" * 50)
print("TEACHER ABLATION")
print("=" * 50)
teacher_ablation = run_cluster_ablation(
    TEACHER_MODEL, tokenizer, class_to_problems,
    teacher_clusters, CLASS_CLUSTERS_TEACHER,
)

## 10. Cluster Pairing

Pair student clusters with teacher clusters per subclass by matching
normalized ablation importance:  `Δ(s,c) = baseline - acc_after_ablation`.
Each student cluster is matched to the teacher cluster with the closest Δ.

In [ ]:
@dataclass
class ClusterMapping:
    subclass: int
    student_cluster_idx: int
    teacher_cluster_idx: int
    student_importance: float
    teacher_importance: float
    distance: float


@dataclass
class ClusterPairInfo:
    """Cluster pair enriched with neuron indices for distillation."""
    subclass: int
    student_cluster_idx: int
    teacher_cluster_idx: int
    student_neuron_indices: torch.Tensor
    teacher_neuron_indices: torch.Tensor
    importance: float


def ablation_to_importance(ablation_data):
    """Convert ablation_performance.json into {subclass: {cluster: importance}}."""
    result = {}
    for sc_str, entry in ablation_data.items():
        baseline = entry["baseline"]
        inner = {}
        for c_str, acc in entry["clusters"].items():
            drop = baseline - acc
            inner[int(c_str)] = max(drop, 0.0)
        result[int(sc_str)] = inner
    return result


def normalize_scores(scores):
    out = {}
    for sc, cs in scores.items():
        mx = max(cs.values()) if cs else 1.0
        if mx == 0:
            out[sc] = dict(cs)
        else:
            out[sc] = {c: v / mx for c, v in cs.items()}
    return out


def pair_clusters(
    delta_s, delta_t,
    student_clusters, teacher_clusters,
    top_k_per_subclass=None,
):
    """Create ClusterPairInfo list from ablation importance scores."""
    delta_s_n = normalize_scores(delta_s)
    delta_t_n = normalize_scores(delta_t)

    pairs = []
    common = sorted(set(delta_s_n) & set(delta_t_n))

    for sc in common:
        s_scores = delta_s_n[sc]
        t_scores = delta_t_n[sc]
        if not s_scores or not t_scores:
            continue

        if top_k_per_subclass is not None:
            sorted_s = sorted(s_scores.items(), key=lambda x: x[1], reverse=True)
            s_scores = dict(sorted_s[:top_k_per_subclass])

        for s_cid, s_imp in s_scores.items():
            best_t = min(t_scores, key=lambda t: abs(s_imp - t_scores[t]))
            t_imp = t_scores[best_t]
            dist = abs(s_imp - t_imp)

            if sc not in student_clusters or sc not in teacher_clusters:
                continue
            if s_cid not in student_clusters[sc] or best_t not in teacher_clusters[sc]:
                continue

            s_idx = student_clusters[sc][s_cid]
            t_idx = teacher_clusters[sc][best_t]

            if not isinstance(s_idx, torch.Tensor):
                s_idx = torch.tensor(s_idx, dtype=torch.long)
            if not isinstance(t_idx, torch.Tensor):
                t_idx = torch.tensor(t_idx, dtype=torch.long)

            if s_idx.numel() == 0 or t_idx.numel() == 0:
                continue

            pairs.append(ClusterPairInfo(
                subclass=sc,
                student_cluster_idx=s_cid,
                teacher_cluster_idx=best_t,
                student_neuron_indices=s_idx,
                teacher_neuron_indices=t_idx,
                importance=s_imp,
            ))

    pairs.sort(key=lambda p: p.importance, reverse=True)
    return pairs


# --- Build cluster pairs ---
delta_s = ablation_to_importance(student_ablation)
delta_t = ablation_to_importance(teacher_ablation)

cluster_pairs = pair_clusters(
    delta_s, delta_t,
    student_clusters, teacher_clusters,
    top_k_per_subclass=TOP_K_CLUSTERS,
)

print(f"\nTotal cluster pairs: {len(cluster_pairs)}")
print(f"\nPair details:")
for p in cluster_pairs:
    print(
        f"  Subclass {p.subclass}: "
        f"S_cluster {p.student_cluster_idx} ({p.student_neuron_indices.numel()} neurons) -> "
        f"T_cluster {p.teacher_cluster_idx} ({p.teacher_neuron_indices.numel()} neurons)  "
        f"importance={p.importance:.4f}"
    )

# Save mapping
mapping_path = os.path.join(SAVE_DIR, "cluster_mapping.json")
os.makedirs(SAVE_DIR, exist_ok=True)
mapping_data = [
    {
        "subclass": p.subclass,
        "student_cluster": p.student_cluster_idx,
        "teacher_cluster": p.teacher_cluster_idx,
        "student_neurons": p.student_neuron_indices.numel(),
        "teacher_neurons": p.teacher_neuron_indices.numel(),
        "importance": p.importance,
    }
    for p in cluster_pairs
]
with open(mapping_path, "w") as f:
    json.dump(mapping_data, f, indent=2)
print(f"\nSaved cluster mapping to {mapping_path}")

## 11. CKA Loss and Cluster Alignment Loss

In [ ]:
def linear_cka_efficient(X, Y, eps=1e-8):
    """Efficient linear CKA (invariant to feature dimension)."""
    X = X.float()
    Y = Y.float()
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    YtX = Y.T @ X
    XtX = X.T @ X
    YtY = Y.T @ Y

    numerator = (YtX ** 2).sum()
    denominator = torch.sqrt((XtX ** 2).sum() * (YtY ** 2).sum() + eps)
    return torch.clamp(numerator / denominator, 0.0, 1.0)


class ClusterActivationCache:
    """Hooks up_proj in all MLP layers, provides flattened last-token activations.

    Neuron indices from clustering index into the concatenated vector
    [layer_0.up_proj | layer_1.up_proj | ... | layer_N.up_proj].
    """

    def __init__(self):
        self.layer_activations = {}
        self.hooks = []

    def _make_hook(self, layer_idx, detach):
        def hook(module, _input, output):
            self.layer_activations[layer_idx] = output.detach() if detach else output
        return hook

    def register_hooks(self, model, detach=False):
        self.clear()
        for i, layer in enumerate(model.model.layers):
            h = layer.mlp.up_proj.register_forward_hook(
                self._make_hook(i, detach=detach)
            )
            self.hooks.append(h)

    def get_flattened_last_token(self, attention_mask=None):
        """Returns (B, num_layers * intermediate_size) in float32."""
        layers = sorted(self.layer_activations.keys())
        parts = []
        for i in layers:
            act = self.layer_activations[i]  # (B, S, D)
            if attention_mask is not None:
                last_idx = attention_mask.sum(dim=1).long() - 1
                last_tok = act[torch.arange(act.size(0), device=act.device), last_idx]
            else:
                last_tok = act[:, -1, :]
            parts.append(last_tok.float())
        return torch.cat(parts, dim=-1)

    def clear(self):
        self.layer_activations.clear()
        for h in self.hooks:
            h.remove()
        self.hooks.clear()


class ClusterAlignmentLoss(nn.Module):
    """Importance-weighted CKA over paired neuron clusters."""

    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, student_flat, teacher_flat, pairs, importance_weighting=True):
        """
        Args:
            student_flat: (B, D_student_total)
            teacher_flat: (B, D_teacher_total)
            pairs: List[ClusterPairInfo]
            importance_weighting: weight each pair by ablation importance

        Returns:
            (total_loss, cka_scores_dict)
        """
        dev = student_flat.device
        losses, weights = [], []
        cka_scores = {}

        for p in pairs:
            s_idx = p.student_neuron_indices.to(dev)
            t_idx = p.teacher_neuron_indices.to(dev)

            s_act = student_flat[:, s_idx]
            t_act = teacher_flat[:, t_idx]

            if s_act.size(0) < 2:
                continue

            cka = linear_cka_efficient(s_act, t_act, eps=self.eps)
            loss = 1.0 - cka
            losses.append(loss)
            weights.append(p.importance if importance_weighting else 1.0)
            cka_scores[(p.subclass, p.student_cluster_idx, p.teacher_cluster_idx)] = cka.item()

        if not losses:
            return torch.tensor(0.0, device=dev, requires_grad=True), {}

        losses_t = torch.stack(losses)
        weights_t = torch.tensor(weights, device=dev, dtype=losses_t.dtype)
        weights_t = weights_t / (weights_t.sum() + 1e-12)
        total = (losses_t * weights_t).sum()
        return total, cka_scores


class ProjectionHeadBank(nn.Module):
    """Optional learned linear projections for direct cluster alignment."""

    def __init__(self, pairs):
        super().__init__()
        self.projections = nn.ModuleDict()
        self._keys = []
        for p in pairs:
            key = f"s{p.subclass}_c{p.student_cluster_idx}"
            self.projections[key] = nn.Linear(
                p.student_neuron_indices.numel(),
                p.teacher_neuron_indices.numel(),
                bias=False,
            )
            self._keys.append(key)

    def forward(self, student_flat, teacher_flat, pairs):
        dev = student_flat.device
        losses = []
        for p, key in zip(pairs, self._keys):
            s_act = student_flat[:, p.student_neuron_indices.to(dev)]
            t_act = teacher_flat[:, p.teacher_neuron_indices.to(dev)].detach()
            projected = self.projections[key](s_act)
            losses.append(F.mse_loss(projected, t_act))
        if not losses:
            return torch.tensor(0.0, device=dev, requires_grad=True)
        return torch.stack(losses).mean()


print("Loss functions defined.")

## 12. Dataset and Evaluation

In [ ]:
class AddDataset(Dataset):
    def __init__(self, json_data, tokenizer):
        self.data = list(json_data.items())
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prompt, answer = self.data[idx]
        answer = str(answer)
        prompt_ids = self.tokenizer(prompt, return_tensors="pt", padding=False)["input_ids"].squeeze(0)
        answer_ids = self.tokenizer(
            answer + self.tokenizer.eos_token, return_tensors="pt", padding=False,
        )["input_ids"].squeeze(0)

        input_ids = torch.cat([prompt_ids, answer_ids])
        attention_mask = torch.ones_like(input_ids)
        labels = torch.full_like(input_ids, -100)
        labels[len(prompt_ids):] = answer_ids

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "prompt_len": len(prompt_ids),
        }


def extract_int_after_equals(text):
    m = re.search(r"=\s*(\d+)", text)
    return int(m.group(1)) if m else None


@torch.no_grad()
def eval_accuracy(model, tokenizer, data, batch_size=50):
    model.eval()
    prompts = list(data.keys())
    answers = list(data.values())
    correct = total = 0

    orig_side = tokenizer.padding_side
    tokenizer.padding_side = "right"

    for i in tqdm(range(0, len(prompts), batch_size), desc="Eval", leave=False):
        bp = prompts[i : i + batch_size]
        ba = answers[i : i + batch_size]
        inputs = tokenizer(bp, return_tensors="pt", padding=True).to(model.device)
        outputs = model.generate(
            **inputs, max_new_tokens=10, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for pred, gold in zip(decoded, ba):
            if extract_int_after_equals(pred) == gold:
                correct += 1
            total += 1

    tokenizer.padding_side = orig_side
    return correct / max(total, 1)


print("Dataset and eval functions defined.")

## 13. Load Student and Teacher Models

In [ ]:
print(f"Loading student: {STUDENT_MODEL}")
student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=torch.float32, device_map=device,
)

print(f"Loading teacher: {TEACHER_MODEL}")
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL, torch_dtype=torch.float32, device_map=device,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print(f"Student layers: {student.config.num_hidden_layers}, intermediate: {student.config.intermediate_size}")
print(f"Teacher layers: {teacher.config.num_hidden_layers}, intermediate: {teacher.config.intermediate_size}")
print(f"Student total neurons: {student.config.num_hidden_layers * student.config.intermediate_size}")
print(f"Teacher total neurons: {teacher.config.num_hidden_layers * teacher.config.intermediate_size}")

## 14. Baseline Evaluation

In [ ]:
print("=" * 50)
print("BASELINE EVALUATION")
print("=" * 50)

baseline_student = eval_accuracy(student, tokenizer, TEST_DATA)
print(f"Student baseline: {baseline_student:.3f}")

baseline_teacher = eval_accuracy(teacher, tokenizer, TEST_DATA)
print(f"Teacher baseline: {baseline_teacher:.3f}")
print("=" * 50)

## 15. Neuron-Cluster Distillation Training

The training loop:
- Forward pass through teacher (frozen, no grad) -> hook all `up_proj` activations
- Forward pass through student (trainable) -> hook all `up_proj` activations
- Flatten last-token activations across all layers
- For each paired cluster, extract neuron subsets and compute CKA
- Total loss = `L_CE + λ_cluster · Σ w_i · (1 - CKA_i)`
- Backprop through student only

In [ ]:
# Setup training components
train_dataset = AddDataset(TRAIN_DATA, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

student_cache = ClusterActivationCache()
teacher_cache = ClusterActivationCache()

cluster_loss_fn = ClusterAlignmentLoss()

proj_heads = None
if USE_PROJECTION_HEADS and cluster_pairs:
    proj_heads = ProjectionHeadBank(cluster_pairs).to(device)

params = list(student.parameters())
if proj_heads is not None:
    params += list(proj_heads.parameters())
optimizer = AdamW(params, lr=LEARNING_RATE)

print(f"Training samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Cluster pairs for alignment: {len(cluster_pairs)}")
print(f"Projection heads: {USE_PROJECTION_HEADS}")

In [ ]:
history = defaultdict(list)

print(f"\n{'=' * 60}")
print(f"Starting Neuron-Cluster Circuit Distillation")
print(f"  Epochs: {EPOCHS}")
print(f"  LR: {LEARNING_RATE}, lambda_cluster: {LAMBDA_CLUSTER}")
print(f"  Cluster pairs: {len(cluster_pairs)}")
print(f"  Importance weighting: {IMPORTANCE_WEIGHTING}")
print(f"{'=' * 60}\n")

best_accuracy = baseline_student

for epoch in range(EPOCHS):
    student.train()
    epoch_ce, epoch_cluster, epoch_proj, epoch_total = 0, 0, 0, 0
    epoch_cka_sum, n_batches = 0, 0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        if (labels != -100).sum().item() == 0:
            continue

        # Register hooks
        teacher_cache.register_hooks(teacher, detach=True)
        student_cache.register_hooks(student, detach=False)

        try:
            # Teacher forward (no grad)
            with torch.no_grad():
                teacher(input_ids=input_ids, attention_mask=attention_mask)
            t_flat = teacher_cache.get_flattened_last_token(attention_mask)

            # Student forward
            student_out = student(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels,
            )
            s_flat = student_cache.get_flattened_last_token(attention_mask)

            ce_loss = student_out.loss

            # Cluster CKA alignment
            cl_loss, cka_scores = cluster_loss_fn(
                s_flat, t_flat, cluster_pairs,
                importance_weighting=IMPORTANCE_WEIGHTING,
            )

            # Optional projection loss
            p_loss = torch.tensor(0.0, device=device)
            if proj_heads is not None and LAMBDA_PROJ > 0:
                p_loss = proj_heads(s_flat, t_flat, cluster_pairs)

            total_loss = ce_loss + LAMBDA_CLUSTER * cl_loss + LAMBDA_PROJ * p_loss

            if torch.isnan(total_loss):
                continue

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            optimizer.step()

            epoch_ce += ce_loss.item()
            epoch_cluster += cl_loss.item()
            epoch_proj += p_loss.item()
            epoch_total += total_loss.item()
            mean_cka = sum(cka_scores.values()) / len(cka_scores) if cka_scores else 0.0
            epoch_cka_sum += mean_cka
            n_batches += 1

            if step % 50 == 0:
                print(
                    f"  step {step:04d} | "
                    f"CE {ce_loss.item():.4f} | "
                    f"Cluster {cl_loss.item():.4f} | "
                    f"CKA {mean_cka:.4f}"
                )

        finally:
            student_cache.clear()
            teacher_cache.clear()

    # Epoch averages
    if n_batches > 0:
        avg_ce = epoch_ce / n_batches
        avg_cl = epoch_cluster / n_batches
        avg_proj = epoch_proj / n_batches
        avg_total = epoch_total / n_batches
        avg_cka = epoch_cka_sum / n_batches
    else:
        avg_ce = avg_cl = avg_proj = avg_total = avg_cka = 0

    # Evaluation
    accuracy = 0.0
    if (epoch + 1) % EVAL_EVERY == 0:
        accuracy = eval_accuracy(student, tokenizer, TEST_DATA)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        save_path = os.path.join(SAVE_DIR, "best_model")
        os.makedirs(save_path, exist_ok=True)
        student.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  ** New best model saved! (acc={accuracy:.3f})")

    history["epoch"].append(epoch + 1)
    history["ce_loss"].append(avg_ce)
    history["cluster_loss"].append(avg_cl)
    history["proj_loss"].append(avg_proj)
    history["total_loss"].append(avg_total)
    history["mean_cka"].append(avg_cka)
    history["accuracy"].append(accuracy)

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}: "
        f"CE={avg_ce:.4f}, Cluster={avg_cl:.4f}, "
        f"CKA={avg_cka:.4f}, Acc={accuracy:.3f}"
    )

print(f"\n{'=' * 60}")
print(f"Training complete!")
print(f"Best accuracy: {best_accuracy:.3f} (baseline: {baseline_student:.3f})")
print(f"{'=' * 60}")

## 16. Final Evaluation

In [ ]:
print("\n" + "=" * 50)
print("FINAL EVALUATION")
print("=" * 50)

final_accuracy = eval_accuracy(student, tokenizer, TEST_DATA)
print(f"Final student accuracy:    {final_accuracy:.3f}")
print(f"Student baseline:          {baseline_student:.3f}")
print(f"Teacher baseline:          {baseline_teacher:.3f}")
print(f"Improvement over baseline: {(final_accuracy - baseline_student)*100:+.2f}%")
print("=" * 50)

## 17. Training Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Total loss
ax = axes[0, 0]
ax.plot(history["epoch"], history["total_loss"], "b-", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Total Loss")
ax.set_title("Total Loss (CE + λ·Cluster)")
ax.grid(True, alpha=0.3)

# CE and Cluster loss
ax = axes[0, 1]
ax.plot(history["epoch"], history["ce_loss"], "b-o", label="CE Loss", markersize=3)
ax.plot(history["epoch"], history["cluster_loss"], "r-o", label="Cluster CKA Loss", markersize=3)
if any(v > 0 for v in history["proj_loss"]):
    ax.plot(history["epoch"], history["proj_loss"], "g-o", label="Proj Loss", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Loss Components")
ax.legend()
ax.grid(True, alpha=0.3)

# Mean CKA
ax = axes[0, 2]
ax.plot(history["epoch"], history["mean_cka"], "m-o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean CKA")
ax.set_title("Cluster CKA Alignment")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1, 0]
ax.plot(history["epoch"], history["accuracy"], "g-o", markersize=4)
ax.axhline(y=baseline_student, color="gray", linestyle="--", label=f"Student baseline ({baseline_student:.3f})")
ax.axhline(y=baseline_teacher, color="blue", linestyle=":", alpha=0.5, label=f"Teacher ({baseline_teacher:.3f})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Student Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# Improvement over baseline
ax = axes[1, 1]
improvement = [a - baseline_student for a in history["accuracy"]]
colors = ["green" if x > 0 else "red" for x in improvement]
ax.bar(history["epoch"], improvement, color=colors)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy Change")
ax.set_title("Accuracy Improvement vs Baseline")
ax.grid(True, alpha=0.3)

# CKA vs Accuracy correlation
ax = axes[1, 2]
valid = [(c, a) for c, a in zip(history["mean_cka"], history["accuracy"]) if a > 0]
if valid:
    ckas, accs = zip(*valid)
    ax.scatter(ckas, accs, c="purple", alpha=0.6)
    ax.set_xlabel("Mean Cluster CKA")
    ax.set_ylabel("Accuracy")
    ax.set_title("CKA vs Accuracy")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig_path = os.path.join(SAVE_DIR, "training_results.png")
plt.savefig(save_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_fig_path}")

## 18. Save Results

In [ ]:
# Save final model
final_save_path = os.path.join(SAVE_DIR, "final_model")
os.makedirs(final_save_path, exist_ok=True)
student.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

# Save training history and config
results = {
    "config": {
        "teacher": TEACHER_MODEL,
        "student": STUDENT_MODEL,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "lambda_cluster": LAMBDA_CLUSTER,
        "lambda_proj": LAMBDA_PROJ,
        "top_k_clusters": TOP_K_CLUSTERS,
        "importance_weighting": IMPORTANCE_WEIGHTING,
        "use_projection_heads": USE_PROJECTION_HEADS,
        "num_cluster_pairs": len(cluster_pairs),
    },
    "baseline_student": baseline_student,
    "baseline_teacher": baseline_teacher,
    "final_accuracy": final_accuracy,
    "best_accuracy": best_accuracy,
    "history": dict(history),
}

results_path = os.path.join(SAVE_DIR, "training_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Model saved to: {final_save_path}")
print(f"Results saved to: {results_path}")

## 19. Ablation Importance Visualization

Visualize per-cluster ablation importance for both models, and the
resulting cluster pairings.

In [ ]:
n_subclasses = len(set(p.subclass for p in cluster_pairs))
if n_subclasses > 0:
    fig, axes = plt.subplots(1, min(n_subclasses, 4), figsize=(5 * min(n_subclasses, 4), 5))
    if n_subclasses == 1:
        axes = [axes]

    subclasses_shown = sorted(set(p.subclass for p in cluster_pairs))[:4]

    for ax_idx, sc in enumerate(subclasses_shown):
        ax = axes[ax_idx]

        s_imp = delta_s.get(sc, {})
        t_imp = delta_t.get(sc, {})

        s_keys = sorted(s_imp.keys())
        t_keys = sorted(t_imp.keys())
        s_vals = [s_imp[k] for k in s_keys]
        t_vals = [t_imp[k] for k in t_keys]

        x_s = list(range(len(s_keys)))
        x_t = [x + len(s_keys) + 1 for x in range(len(t_keys))]

        ax.bar(x_s, s_vals, color="steelblue", alpha=0.8, label="Student")
        ax.bar(x_t, t_vals, color="coral", alpha=0.8, label="Teacher")

        ax.set_xticks(x_s + x_t)
        ax.set_xticklabels(
            [f"S{k}" for k in s_keys] + [f"T{k}" for k in t_keys],
            rotation=45, fontsize=8,
        )
        ax.set_ylabel("Importance (accuracy drop)")
        ax.set_title(f"Subclass {sc}")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Draw pairing arrows
        for p in cluster_pairs:
            if p.subclass != sc:
                continue
            if p.student_cluster_idx in s_keys and p.teacher_cluster_idx in t_keys:
                sx = s_keys.index(p.student_cluster_idx)
                tx = len(s_keys) + 1 + t_keys.index(p.teacher_cluster_idx)
                sy = s_imp[p.student_cluster_idx]
                ty = t_imp[p.teacher_cluster_idx]
                ax.annotate(
                    "", xy=(tx, ty), xytext=(sx, sy),
                    arrowprops=dict(arrowstyle="->", color="green", lw=1.5, alpha=0.6),
                )

    plt.tight_layout()
    ablation_fig_path = os.path.join(SAVE_DIR, "cluster_pairing_importance.png")
    plt.savefig(ablation_fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {ablation_fig_path}")
else:
    print("No cluster pairs to visualize.")

## 20. Download Results (Optional, Colab)

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("cluster_distillation_results", "zip", SAVE_DIR)
    files.download("cluster_distillation_results.zip")
except ImportError:
    print("Not running in Colab — skip download.  Results are in:", SAVE_DIR)